# 🦥 Fine-tune & Evaluate Gemma 4 Vision on Google Colab

This notebook provides a complete pipeline for fine-tuning and evaluating the Gemma 4 Vision (E2B) model using Unsloth.

## 1. Install Dependencies
We use Unsloth for 2x faster training and significantly lower VRAM usage.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

In [2]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [3]:
%%capture
!pip install --no-deps --upgrade timm # For Gemma 4 vision/audio

## 2. Mount Google Drive
Mount your drive to access the dataset and save the trained model.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Configuration & Helpers

In [4]:
import os
# IMPORT UNSLOTH FIRST: This is required for speed and memory optimizations
import unsloth
from unsloth import FastVisionModel, get_chat_template
from unsloth.trainer import UnslothVisionDataCollator
import json
import torch
import time
from PIL import Image
from datasets import Dataset
from transformers import TextStreamer
from trl import SFTConfig, SFTTrainer

# Check if GPU is available to avoid the NotImplementedError
if not torch.cuda.is_available():
    print("WARNING: No GPU detected. Unsloth requires a GPU to run. Please change your runtime type to GPU.")

# --- CONFIGURATION ---
LOCAL_BASE_PATH = "/home/quangnhvn34/dev/me/InsureVN/data/health_insurance/health_insurance_extracted"
DATA_DIR = "/content/drive/MyDrive/health_insurance_extracted"

# Save all outputs in the same Google Drive folder as the data
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs_gemma4_finetuned")
SAVE_DIR = os.path.join(DATA_DIR, "gemma4-e2b-finetuned-lora")
GGUF_DIR = os.path.join(DATA_DIR, "gemma4-gguf")

GGUF_METHOD = "q4_k_m"
NUM_EPOCHS = 2

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

def fix_path(old_path):
    """Converts local machine paths to Colab Google Drive paths."""
    if isinstance(old_path, str) and old_path.startswith(LOCAL_BASE_PATH):
        return old_path.replace(LOCAL_BASE_PATH, DATA_DIR)
    basename = os.path.basename(str(old_path))
    return os.path.join(DATA_DIR, basename)

def load_jsonl_as_unsloth_dataset(jsonl_path):
    converted = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            formatted_messages = []
            for msg in data["messages"]:
                if msg["role"] == "user":
                    new_content = []
                    for item in msg["content"]:
                        if item["type"] == "image_path":
                            corrected_image_path = fix_path(item["content"])
                            new_content.append({"type": "image", "image": corrected_image_path})
                        elif item["type"] == "text":
                            new_content.append({"type": "text", "text": item["content"]})
                    formatted_messages.append({"role": "user", "content": new_content})
                elif msg["role"] == "assistant":
                    formatted_messages.append({"role": "assistant", "content": [{"type": "text", "text": msg["content"]}]})
            converted.append({"messages": formatted_messages})
    return converted

def format_user_messages_inference(raw_messages):
    """Convert JSONL user message format to Unsloth inference format."""
    messages = []
    for msg in raw_messages:
        if msg["role"] != "user":
            continue
        new_content = []
        for item in msg["content"]:
            if item["type"] == "image_path":
                new_content.append({"type": "image", "image": fix_path(item["content"])})
            elif item["type"] == "text":
                new_content.append({"type": "text", "text": item["content"]})
        messages.append({"role": "user", "content": new_content})
    return messages

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 4. Load Model & Processor

In [5]:
model, processor = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=32,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
    target_modules="all-linear",
)

processor = get_chat_template(processor, "gemma-4")

==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

## 5. Training

In [ ]:
jsonl_file = os.path.join(DATA_DIR, "oumi_vlm_train.jsonl")
converted_dataset = load_jsonl_as_unsloth_dataset(jsonl_file)
print(f"Loaded {len(converted_dataset)} training samples.")

trainer = SFTTrainer(
    model=model,
    train_dataset=converted_dataset,
    processing_class=processor.tokenizer,
    data_collator=UnslothVisionDataCollator(model, processor),
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.03,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR,
        report_to="none",
        remove_unused_columns=False,
        dataset_text_field="",
        dataset_kwargs={"skip_prepare_dataset": True},
        max_length=2048,
    ),
)

trainer.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loaded 1314 training samples.
Unsloth: Model does not have a default image size - using 512


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,314 | Num Epochs = 2 | Total steps = 330
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,719,680 of 5,182,897,696 (1.15% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Step,Training Loss
1,15.478360
2,14.169378
3,15.358590
4,14.725991
5,14.426007
6,14.109731
7,13.016649
8,11.428267
9,9.227988
10,8.512580


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/health_insurance_extracted/outputs_gemma4_finetuned/checkpoint-330/tokenizer_config.json.


TrainOutput(global_step=330, training_loss=1.1394297107263949, metrics={'train_runtime': 7763.6163, 'train_samples_per_second': 0.339, 'train_steps_per_second': 0.043, 'total_flos': 5.038253733481613e+16, 'train_loss': 1.1394297107263949, 'epoch': 2.0})

## 6. Save & Export

In [ ]:
import os
from unsloth import FastVisionModel, get_chat_template

# Nếu bạn muốn chạy riêng phần này, hãy đảm bảo đã chạy các cell ở mục 1, 2, 3 (Install & Config)
# Script sẽ ưu tiên load model từ OUTPUT_DIR (nơi chứa checkpoints)
if 'model' not in locals():
    print(f"⏳ Đang tải model và adapter từ {OUTPUT_DIR}...")
    model, processor = FastVisionModel.from_pretrained(
        model_name = OUTPUT_DIR, # Load trực tiếp từ folder checkpoint/output
        load_in_4bit = True,
    )
    processor = get_chat_template(processor, "gemma-4")

# Lưu bản chính thức vào SAVE_DIR
model.save_pretrained(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print(f"✅ Adapters đã được lưu gọn gàng vào {SAVE_DIR}")

# Export sang GGUF nếu cần
if GGUF_DIR:
    print(f"🚀 Đang export sang GGUF ({GGUF_METHOD})...")
    model.save_pretrained_gguf(
        GGUF_DIR,
        processor.tokenizer,
        quantization_method=GGUF_METHOD,
    )
    print(f"✅ GGUF model đã sẵn sàng tại {GGUF_DIR}")


In [ ]:
import gc
import torch

# 1. Giải phóng bộ nhớ
gc.collect()
torch.cuda.empty_cache()

try:
    print(f"⏳ Đang thử lưu lại adapter vào {SAVE_DIR}...")
    # Thử lưu lại một lần nữa sau khi đã dọn dẹp cache
    model.save_pretrained(SAVE_DIR)
    processor.save_pretrained(SAVE_DIR)
    print("✅ Đã lưu thành công sau khi dọn dẹp bộ nhớ!")
except Exception as e:
    print(f"❌ Vẫn gặp lỗi khi lưu: {e}")
    print("💡 Lời khuyên: Nếu lỗi 357 vs 317 vẫn tiếp diễn, bạn hãy thử thay 'target_modules' trong cell 2AZe4SRXtgzj thành các module cụ thể như ['q_proj', 'k_proj', 'v_proj', 'o_proj'] thay vì 'all-linear'.")

## 7. Evaluation
Run inference on test samples to verify performance.

In [6]:
# Switch to inference mode
# Chuyển sang chế độ Inference
if 'model' in locals():
    print("⚡ Đang kích hoạt chế độ Fast Inference...")
    FastVisionModel.for_inference(model)
else:
    print("⚠️ Cảnh báo: Biến 'model' chưa được load. Vui lòng chạy cell ở Phần 6 trước để load model.")

test_file = os.path.join(DATA_DIR, "oumi_vlm_test.jsonl")
test_examples = []
with open(test_file, "r", encoding="utf-8") as f:
    for line in f:
        test_examples.append(json.loads(line))

print(f"✅ Đã tải {len(test_examples)} mẫu test từ {test_file}")

FastVisionModel.for_inference(model)

test_file = os.path.join(DATA_DIR, "oumi_vlm_test.jsonl")
test_examples = []
with open(test_file, "r", encoding="utf-8") as f:
    for line in f:
        test_examples.append(json.loads(line))

print(f"Loaded {len(test_examples)} test samples.")

def run_eval(num_samples=3):
    for i in range(min(num_samples, len(test_examples))):
        sample = test_examples[i]
        messages = format_user_messages_inference(sample["messages"])

        # Extract image path
        image_path = next((item["image"] for item in messages[0]["content"] if item["type"] == "image"), None)
        if not image_path or not os.path.exists(image_path):
            print(f"Image not found: {image_path}")
            continue

        print(f"\n--- Sample {i+1} ---")
        image = Image.open(image_path)
        input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

        print("Assistant Response:")
        text_streamer = TextStreamer(processor, skip_prompt=True)
        _ = model.generate(
            **inputs,
            streamer=text_streamer,
            max_new_tokens=1024,
            use_cache=True,
            temperature=1.0,
            top_p=0.95,
            top_k=64
        )

        # Show Ground Truth
        ground_truth = next((msg["content"] for msg in sample["messages"] if msg["role"] == "assistant"), "None")
        print(f"\nGround Truth:\n{ground_truth}")
        print("-" * 40)



⚡ Đang kích hoạt chế độ Fast Inference...
✅ Đã tải 146 mẫu test từ /content/drive/MyDrive/health_insurance_extracted/oumi_vlm_test.jsonl
Loaded 146 test samples.


In [ ]:
run_eval(10)


--- Sample 1 ---
Assistant Response:
{
  "has_content": true,
  "content_type": "text",
  "markdown": "CALL CENTER\n1800 - 58 88 12\nMiễn phí 24/7 toàn quốc",
  "structured_data": [],
  "description": "Đây là thông tin về số điện thoại tổng đài (Call Center) của một dịch vụ, với số điện thoại là 1800-58 88 12, miễn phí 24/7 toàn quốc."
}<turn|>

Ground Truth:
{
  "has_content": true,
  "content_type": "text",
  "markdown": "CALL CENTER\n1800 - 58 88 12\nMiễn phí 24/7 toàn quốc",
  "structured_data": [],
  "description": "Hình ảnh biểu tượng Call Center với số điện thoại hotline 1800-58 88 12, miễn phí 24/7 toàn quốc."
}
----------------------------------------

--- Sample 2 ---
Assistant Response:
{
  "has_content": false,
  "content_type": "image_only",
  "markdown": "",
  "structured_data": [],
  "description": "Hình ảnh là một biểu tượng đồ họa gồm ba hình người và một mũi tên chỉ hướng sang phải."
}<turn|>

Ground Truth:
{
  "has_content": true,
  "content_type": "image_only",
  "

In [7]:
!pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 28.9 MB/s eta 0:00:00


In [ ]:
!pip install python-Levenshtein

In [10]:
import json
import os
import time
import pandas as pd
from Levenshtein import distance as levenshtein_distance
from PIL import Image
from unsloth import FastVisionModel, get_chat_template

def calculate_anls(s1, s2, threshold=0.5):
    """Tính toán Average Normalized Levenshtein Similarity."""
    s1, s2 = str(s1).lower().strip(), str(s2).lower().strip()
    if not s1 and not s2: return 1.0
    if not s1 or not s2: return 0.0

    dist = levenshtein_distance(s1, s2)
    max_len = max(len(s1), len(s2))
    score = 1 - (dist / max_len)
    return score if score >= threshold else 0.0

def run_benchmark(num_samples=None):
    global model, processor

    # Tự động load model nếu chưa có trong bộ nhớ
    if 'model' not in globals() or model is None:
        print(f"⏳ Đang tải model từ {SAVE_DIR}...")
        model, processor = FastVisionModel.from_pretrained(
            model_name = SAVE_DIR,
            load_in_4bit = True,
        )
        processor = get_chat_template(processor, "gemma-4")

    print("⚡ Đang kích hoạt chế độ Fast Inference...")
    FastVisionModel.for_inference(model)

    test_file = os.path.join(DATA_DIR, "oumi_vlm_test.jsonl")
    with open(test_file, "r", encoding="utf-8") as f:
        all_test_data = [json.loads(line) for line in f]

    # Nếu num_samples là None, sẽ lấy toàn bộ dataset
    samples = all_test_data[:num_samples] if num_samples else all_test_data
    print(f"🚀 Bắt đầu Benchmark tập trung vào trường 'markdown' trên {len(samples)} mẫu...")

    json_valid_count = 0
    markdown_scores = []
    total_anls = 0

    for i, sample in enumerate(samples):
        messages = format_user_messages_inference(sample["messages"])
        image_path = next((item["image"] for item in messages[0]["content"] if item["type"] == "image"), None)
        ground_truth_str = next((msg["content"] for msg in sample["messages"] if msg["role"] == "assistant"), "{}")

        try:
            gt_json = json.loads(ground_truth_str)
            gt_markdown = gt_json.get("markdown", "")
        except:
            gt_markdown = ""

        # 1. Run Inference
        image = Image.open(image_path)
        input_text = processor.apply_chat_template(messages, add_generation_prompt=True)
        inputs = processor(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

        output_ids = model.generate(**inputs, max_new_tokens=1024, use_cache=True, temperature=0.1)
        response = processor.batch_decode(output_ids[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]

        # 2. Parse Markdown from Response
        pred_markdown = ""
        is_valid_json = False
        try:
            json_start = response.find('{')
            json_end = response.rfind('}') + 1
            if json_start != -1 and json_end != -1:
                pred_json = json.loads(response[json_start:json_end])
                pred_markdown = pred_json.get("markdown", "")
                is_valid_json = True
                json_valid_count += 1
            else:
                pred_markdown = response
        except:
            pred_markdown = response

        # 3. Tính ANLS cho trường Markdown
        mk_score = calculate_anls(pred_markdown, gt_markdown)
        markdown_scores.append(mk_score)
        total_anls += calculate_anls(response, ground_truth_str)

        if (i + 1) % 5 == 0 or (i + 1) == len(samples):
            print(f"[{i+1}/{len(samples)}] Markdown Acc hiện tại: {(sum(markdown_scores)/len(markdown_scores)):.2f}")

    avg_markdown_acc = (sum(markdown_scores) / len(markdown_scores)) * 100
    print("\n" + "="*50)
    print("📊 KẾT QUẢ BENCHMARK TOÀN BỘ TẬP TEST")
    print("="*50)
    print(f"🎯 Độ chính xác Markdown trung bình: {avg_markdown_acc:.1f}%")
    print(f"✅ Tỉ lệ JSON hợp lệ: {(json_valid_count/len(samples))*100:.1f}%")
    print(f"ℹ️ Mean ANLS (Toàn bộ chuỗi): {(total_anls/len(samples))*100:.1f}%")

# Chạy benchmark trên toàn bộ tập test (num_samples=None)
run_benchmark(None)

⚡ Đang kích hoạt chế độ Fast Inference...
🚀 Bắt đầu Benchmark tập trung vào trường 'markdown' trên 146 mẫu...
[5/146] Markdown Acc hiện tại: 0.80
[10/146] Markdown Acc hiện tại: 0.76
[15/146] Markdown Acc hiện tại: 0.77
[20/146] Markdown Acc hiện tại: 0.80
[25/146] Markdown Acc hiện tại: 0.78
[30/146] Markdown Acc hiện tại: 0.75
[35/146] Markdown Acc hiện tại: 0.75
[40/146] Markdown Acc hiện tại: 0.78
[45/146] Markdown Acc hiện tại: 0.77
[50/146] Markdown Acc hiện tại: 0.77
[55/146] Markdown Acc hiện tại: 0.77
[60/146] Markdown Acc hiện tại: 0.75
[65/146] Markdown Acc hiện tại: 0.74
[70/146] Markdown Acc hiện tại: 0.73
[75/146] Markdown Acc hiện tại: 0.70
[80/146] Markdown Acc hiện tại: 0.68
[85/146] Markdown Acc hiện tại: 0.70
[90/146] Markdown Acc hiện tại: 0.69
[95/146] Markdown Acc hiện tại: 0.68
[100/146] Markdown Acc hiện tại: 0.67
[105/146] Markdown Acc hiện tại: 0.66
[110/146] Markdown Acc hiện tại: 0.66
[115/146] Markdown Acc hiện tại: 0.65
[120/146] Markdown Acc hiện tại: 0.6